# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the [FAIR^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library, referencing core entities by their `@id`.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
import warnings
warnings.filterwarnings("ignore")

# Define the dataset URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata and initialize the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}\n")
print(f"Dataset Identifier: {metadata.identifier}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

The following cells will:
* List all record sets (`cr:RecordSet`) in the dataset and their `@id`.
* For each record set, display its fields referencing their own `@id` and column mappings.


In [ ]:
from pprint import pprint

# List all record sets and their fields (referenced by @id)
rs_objs = metadata.record_sets
print(f"Found {len(rs_objs)} record set(s):\n")

for record_set in rs_objs:
    print(f"- RecordSet @id: {record_set.id}")
    if record_set.name:
        print(f"  Name: {record_set.name}")
    if record_set.description:
        print(f"  Description: {record_set.description}")
    print("  Fields:")
    for field in record_set.fields:
        print(f"    - Field @id: {field.id}")
          
        if field.name:
            print(f"      Name: {field.name}")
        if hasattr(field, 'columns') and field.columns:
            for col in field.columns:
                print(f"      Column @id: {col.id} (maps to: {col.name})")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Each record set and field is referenced by its `@id` as above.

We'll extract all record sets present in the dataset. Edit the following variables if you'd like to choose a different record set for further exploration.

In [ ]:
# Get the list of all record set @id(s)
record_sets_ids = [rs.id for rs in metadata.record_sets]

dataframes = {}
for rs_id in record_sets_ids:
    try:
        records = list(dataset.records(record_set=rs_id))
        dataframes[rs_id] = pd.DataFrame(records)
        print(f"Loaded DataFrame for: {rs_id} ({len(records)} records)")
        print(dataframes[rs_id].columns.tolist())
        display(dataframes[rs_id].head())
    except Exception as e:
        print(f"Could not load records for {rs_id}: {e}")
        dataframes[rs_id] = None

## 4. Exploratory Data Analysis (EDA)
Apply data processing: filtering, normalization, and grouping.

In this example, we focus on the main record set (the one with the most records loaded) and its numeric fields.

In [ ]:
import numpy as np
# Select the record set with the largest DataFrame
main_rs_id = max(
    [k for k, v in dataframes.items() if isinstance(v, pd.DataFrame) and not v.empty],
    key=lambda k: len(dataframes[k])
)
main_df = dataframes[main_rs_id]
print(f"Main record set selected: {main_rs_id} ({len(main_df)} records)")

# Find numeric fields - we'll infer by pandas dtype:
numeric_fields = main_df.select_dtypes(include=[np.number]).columns.tolist()
print(f"Numeric fields detected: {numeric_fields}")

if numeric_fields:
    numeric_field = numeric_fields[0]
    threshold = main_df[numeric_field].quantile(0.9)  # Example: Outlier threshold at 90th percentile
    filtered_df = main_df[main_df[numeric_field] <= threshold].copy()
    print(f"Filtered records where {numeric_field} <= {threshold:.2f} (remove top 10% outliers)")
    
    # Normalization
    filtered_df[f"{numeric_field}_normalized"] = (
        filtered_df[numeric_field] - filtered_df[numeric_field].mean()
    ) / filtered_df[numeric_field].std()
    print(f"Normalized field: {numeric_field}")
    
    # Try grouping by first non-numeric field
    group_fields = [col for col in main_df.columns if col not in numeric_fields]
    if group_fields:
        group_field = group_fields[0]
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame('mean').reset_index()
        print(f"Grouped data by {group_field} (mean of {numeric_field}):")
        display(grouped_df.head())
else:
    print("No numeric fields found for EDA.")

## 5. Visualization
Visualize a distribution of a numeric field and relationship with a group variable, if applicable.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_fields:
    # Histogram of the numeric field
    plt.figure(figsize=(8, 5))
    sns.histplot(filtered_df[numeric_field], bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field} (filtered)")
    plt.xlabel(numeric_field)
    plt.show()

    if group_fields:
        # Barplot: mean numeric_field by group_field
        plt.figure(figsize=(10,5))
        order = grouped_df.sort_values('mean', ascending=False)[group_field]
        sns.barplot(data=grouped_df, x=group_field, y='mean', order=order)
        plt.title(f"Mean {numeric_field} by {group_field}")
        plt.ylabel(f"Mean {numeric_field}")
        plt.xticks(rotation=45, ha='right')
        plt.show()
else:
    print("No visualization available: no numeric fields.")

## 6. Conclusion
This notebook demonstrated how to explore the FAIR^2 dataset via `mlcroissant` using entity `@id` references.

Key findings:
- Loaded all record sets using their `@id`.
- Identified primary numeric and group fields for EDA and visualization.
- Provided code structure for flexible field selection and downstream analysis.

You can adapt this notebook to further explore or process other fields and record sets. To learn more, see the Croissant schema at [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json).